# ELLSA 재현 환경 구축 (Colab Pro+)

이 노트북은 위에서부터 순서대로 실행하면 됩니다.

| 파트 | 내용 | 소요 시간 (예상) |
|---|---|---|
| **A** | GPU 확인, 경로 설정, HuggingFace 토큰, **가중치 다운로드 시작 (백그라운드)** | 5분 |
| **B** | Python 3.10 환경 설치, ELLSA·LIBERO 코드, 렌더링 점검 (다운로드와 동시에 진행) | 15–25분 |
| **C** | 다운로드 완료 후 점검, 경로 보정, **스모크 테스트 (LIBERO-Goal 과제 1개 × 2 에피소드)** | 20–40분 |

### 시작 전 체크리스트
- [ ] **런타임 → 런타임 유형 변경**: `A100 GPU` 또는 `H100 GPU`, **고용량 RAM** 켜기
- [ ] (권장) HuggingFace에서 [`meta-llama/Llama-3.1-8B-Instruct`](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) 접근 승인 신청. **승인이 pending이어도 진행 가능**: 다운로드 스크립트가 같은 가중치의 공개 미러 `unsloth/Meta-Llama-3.1-8B-Instruct`로 자동 전환합니다.
- [ ] HuggingFace **Access Token (read)** 발급 → 왼쪽 🔑 **Secrets**에 이름 `HF_TOKEN`으로 저장, "노트북 액세스" 켜기

### 이 노트북이 미리 처리해 둔 문제들 (사전 검증 완료)
- ELLSA `requirements.txt`에 빠진 패키지(`pytorch-lightning`, `einops` 등)를 추가했고, 시뮬레이터 호환을 위해 `numpy==1.26.4`, `mujoco==2.3.7`, `opencv-python==4.11.0.86`으로 고정했습니다. Python 3.10에서 설치와 import를 확인했습니다.
- `flash-attn`은 소스 빌드(수 시간) 대신 **공식 사전 빌드 wheel**을 받습니다.
- 공식 `setup_libero.sh`의 경로 버그(`${pwd}`)를 피해 LIBERO를 직접 설치합니다. 첫 import 때 입력을 기다리며 멈추는 문제도 설정 파일을 미리 만들어 막았습니다.
- 공식 평가 스크립트는 에피소드 수가 50으로 고정되어 있어서, **원본은 그대로 두고** `--num_trials`, `--task_ids`, `--results_jsonl` 옵션을 추가한 사본(`evaluate_libero_smoke.py`)을 만듭니다.
- UniVLA는 평가 시 설정과 토크나이저 파일만 쓰므로 **가중치는 받지 않습니다**. 파트 C에서 ELLSA 체크포인트를 열어 이 가정을 자동으로 검증합니다.

## 파트 A. 확인 → 설정 → 다운로드 시작

In [ ]:
#@title A1. GPU / RAM / 디스크 확인
import subprocess, shutil, psutil
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
gpu_mem_gb = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                                capture_output=True, text=True).stdout.split()[0]) / 1024
print(f"GPU memory : {gpu_mem_gb:.0f} GB")
print(f"System RAM : {psutil.virtual_memory().total/1e9:.0f} GB")
print(f"Local disk : {shutil.disk_usage('/content').free/1e9:.0f} GB free")
cc = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
                    capture_output=True, text=True).stdout.strip()
print("GPU / compute capability:", cc)
if cc and float(cc.split(",")[-1]) >= 10:
    print("\n❌ Blackwell 계열 GPU(G4 = RTX PRO 6000 등)입니다. ELLSA 공식 환경(torch 2.4 + CUDA 12.4 + flash-attn 2.7.4)은 "
          "이 GPU를 지원하지 않아 모델 실행 단계에서 실패합니다. 런타임 유형을 'A100 GPU'로 바꾸세요.")
elif gpu_mem_gb < 70:
    print("\n⚠️  80GB급 GPU가 아닙니다. ELLSA(8B 백본 2개)는 40GB에서 OOM이 날 수 있습니다. "
          "런타임 유형을 A100(80GB) 또는 H100으로 바꾸는 것을 권장합니다.")
else:
    print("\n✅ 80GB급 GPU")

In [ ]:
#@title A2. Drive 연결 + 경로 설정 (모든 경로를 여기서만 정의)
import os
from google.colab import drive
drive.mount('/content/drive')

CACHE_TO_DRIVE = False  #@param {type:"boolean"}
# False(권장): 가중치는 로컬 디스크(/content)에만 받습니다. 빠르고 Drive 용량이 필요 없지만, 세션마다 다시 받습니다(10–30분).
# True: 다 받은 뒤 Drive에도 복사해 두고, 다음 세션에는 Drive에서 복원합니다. Drive 여유 공간 약 60GB 이상이 필요합니다.

P = dict(
    ELLSA_BASE_PATH = "/content/ELLSA",
    LIBERO_ROOT     = "/content/LIBERO",
    VENV            = "/content/venv",
    SCRIPTS         = "/content/ellsa_scripts",
    CKPT_DIR        = "/content/ckpt",
    DATA_DIR        = "/content/data",
    DRIVE_ROOT      = "/content/drive/MyDrive/ellsa",
)
P["LOG_DIR"]      = P["DRIVE_ROOT"] + "/logs"
P["RESULTS_DIR"]  = P["DRIVE_ROOT"] + "/results"
P["DRIVE_CACHE"]  = P["DRIVE_ROOT"] + "/cache"
P["DETECTED_ENV"] = "/content/ellsa_detected.env"
P["CACHE_TO_DRIVE"] = "1" if CACHE_TO_DRIVE else "0"
# paths the ELLSA code reads from environment variables
P["LLAMA_CKPT_PATH"]  = P["CKPT_DIR"] + "/Llama-3.1-8B-Instruct"
P["COSY_CKPT_PATH"]   = P["CKPT_DIR"] + "/CosyVoice2-0.5B"
P["VISION_VQ_PATH"]   = P["CKPT_DIR"] + "/Emu3-VisionTokenizer"
P["UNIVLA_CKPT_PATH"] = P["CKPT_DIR"] + "/UniVLA/UNIVLA_LIBERO_VIDEO_BS192_8K"
P["ELLSA_DATA_PATH"]  = P["DATA_DIR"] + "/ELLSA_test_data"
P["ELLSA_CKPT"]       = P["CKPT_DIR"] + "/ELLSA"          # refined in C1
P["PYTHONPATH"]       = P["LIBERO_ROOT"] + ":" + P["ELLSA_BASE_PATH"] + "/reference/RoboVLMs"
P["MUJOCO_GL"] = "egl"; P["PYOPENGL_PLATFORM"] = "egl"; P["MUJOCO_EGL_DEVICE_ID"] = "0"   # refined in B4
P["TOKENIZERS_PARALLELISM"] = "false"
# Colab sets MPLBACKEND to its notebook-only backend, which does not exist inside the venv
# (matplotlib then fails to import, and lightning with it). Agg works everywhere.

for k in ["LOG_DIR", "RESULTS_DIR", "SCRIPTS", "CKPT_DIR", "DATA_DIR"] + (["DRIVE_CACHE"] if CACHE_TO_DRIVE else []):
    os.makedirs(P[k], exist_ok=True)
os.environ.update(P)

# bash cells source this file: `source /content/ellsa_env.sh`
with open("/content/ellsa_env.sh", "w") as f:
    for k, v in P.items():
        f.write(f'export {k}="{v}"\n')
    f.write('export PATH="$VENV/bin:$PATH"\n')
    f.write('export MPLBACKEND=Agg\n')
    f.write('[ -f /content/ellsa_detected.env ] && source /content/ellsa_detected.env\n')
    f.write('[ -f /content/ellsa_render.env ] && source /content/ellsa_render.env\n')

free = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"Drive free: {free:.0f} GB" + ("   ⚠️ CACHE_TO_DRIVE에는 60GB 이상 권장" if CACHE_TO_DRIVE and free < 60 else ""))
print("Logs/results ->", P["DRIVE_ROOT"])

In [ ]:
#@title A3. HuggingFace 토큰 + Llama 접근 권한 확인
import os, subprocess
subprocess.run(["pip", "install", "-q", "-U", "huggingface_hub>=0.30", "hf_xet"], check=False)
from google.colab import userdata
from huggingface_hub import hf_hub_download
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN loaded from Colab Secrets")
except Exception as e:
    print("❌ Secrets에 HF_TOKEN이 없습니다. 왼쪽 🔑 아이콘에서 추가하고 '노트북 액세스'를 켠 뒤 다시 실행하세요.", e)
try:
    hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "config.json", token=os.environ.get("HF_TOKEN"),
                    local_dir="/tmp/llama_access_check")
    print("✅ Llama-3.1-8B-Instruct 접근 가능")
except Exception as e:
    print("⚠️ 공식 Llama 저장소 접근 승인 대기 중 → 다운로드 스크립트가 같은 가중치의 공개 미러(unsloth/Meta-Llama-3.1-8B-Instruct)로 자동 전환합니다. 기다리지 않아도 됩니다.\n  ", str(e)[:200])

### A4. 헬퍼 스크립트 생성
아래 다섯 셀은 파일을 만들기만 합니다. 실행만 하세요.

In [ ]:
%%writefile /content/ellsa_scripts/download_weights.py
"""Download every checkpoint and dataset ELLSA evaluation needs.

Runs in the background from the Colab notebook. Writes progress to
$LOG_DIR/download.log and a machine-readable $LOG_DIR/download_status.json.

Usage:
    python download_weights.py                 # all default jobs
    python download_weights.py UniVLA-weights  # only the named job(s)

Environment variables (set by the notebook):
    CKPT_DIR        local folder for checkpoints   (e.g. /content/ckpt)
    DATA_DIR        local folder for test data     (e.g. /content/data)
    LOG_DIR         folder for logs/status         (e.g. /content/drive/MyDrive/ellsa/logs)
    DRIVE_CACHE     Drive folder used as a cache   (e.g. /content/drive/MyDrive/ellsa/cache)
    CACHE_TO_DRIVE  "1" to copy finished downloads to Drive and restore from there next time
    HF_TOKEN        HuggingFace token (needed for the gated Llama repo)
"""
import json
import os
import shutil
import sys
import time
import traceback

from huggingface_hub import snapshot_download

CKPT_DIR = os.environ["CKPT_DIR"]
DATA_DIR = os.environ["DATA_DIR"]
LOG_DIR = os.environ["LOG_DIR"]
DRIVE_CACHE = os.environ.get("DRIVE_CACHE", "")
CACHE_TO_DRIVE = os.environ.get("CACHE_TO_DRIVE", "0") == "1"
TOKEN = os.environ.get("HF_TOKEN") or None

WEIGHT_FILES = ["*.safetensors", "*.bin", "*.pt", "*.pth", "*.ckpt", "*.msgpack", "*.h5"]

# Order matters: the smallest / most essential first, the gated Llama last.
JOBS = [
    dict(name="ELLSA_test_data", repo="tsinghua-ee/ELLSA_test_data", repo_type="dataset",
         dest=os.path.join(DATA_DIR, "ELLSA_test_data")),
    dict(name="ELLSA", repo="tsinghua-ee/ELLSA",
         dest=os.path.join(CKPT_DIR, "ELLSA")),
    dict(name="Emu3-VisionTokenizer", repo="BAAI/Emu3-VisionTokenizer",
         dest=os.path.join(CKPT_DIR, "Emu3-VisionTokenizer")),
    # At evaluation time ELLSA only reads config.json + tokenizer files from the UniVLA
    # folder (the vision expert weights come from the ELLSA checkpoint), so weights are skipped.
    dict(name="UniVLA", repo="Yuqi1997/UniVLA",
         allow=["UNIVLA_LIBERO_VIDEO_BS192_8K/*"], ignore=WEIGHT_FILES,
         dest=os.path.join(CKPT_DIR, "UniVLA")),
    dict(name="CosyVoice2-0.5B", repo="FunAudioLLM/CosyVoice2-0.5B",
         dest=os.path.join(CKPT_DIR, "CosyVoice2-0.5B")),
    # "original/*" is a duplicate consolidated .pth copy (~16 GB) that HF transformers does not need.
    # If access to the gated Meta repo is still pending, fall back to an ungated mirror with the
    # same weights (same Llama 3.1 Community License applies). LLAMA_REPO overrides the first choice.
    dict(name="Llama-3.1-8B-Instruct",
         repo=os.environ.get("LLAMA_REPO", "meta-llama/Llama-3.1-8B-Instruct"),
         fallback_repo="unsloth/Meta-Llama-3.1-8B-Instruct",
         ignore=["original/*"], dest=os.path.join(CKPT_DIR, "Llama-3.1-8B-Instruct")),
]
# Optional job, only run when asked for by name (see inspect_assets.py verdict).
OPTIONAL = [
    dict(name="UniVLA-weights", repo="Yuqi1997/UniVLA",
         allow=["UNIVLA_LIBERO_VIDEO_BS192_8K/*"],
         dest=os.path.join(CKPT_DIR, "UniVLA")),
]

STATUS_PATH = os.path.join(LOG_DIR, "download_status.json")


def load_status():
    try:
        with open(STATUS_PATH) as f:
            return json.load(f)
    except Exception:
        return {}


def save_status(status):
    tmp = STATUS_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(status, f, indent=1)
    os.replace(tmp, STATUS_PATH)


def dir_size_gb(path):
    total = 0
    for root, _, files in os.walk(path):
        for fn in files:
            fp = os.path.join(root, fn)
            if not os.path.islink(fp):
                try:
                    total += os.path.getsize(fp)
                except OSError:
                    pass
    return round(total / 1e9, 2)


def log(msg):
    print(time.strftime("[%H:%M:%S] ") + msg, flush=True)


def run_job(job, status):
    name, dest = job["name"], job["dest"]
    marker = os.path.join(dest, ".download_complete")
    cache_dir = os.path.join(DRIVE_CACHE, name) if DRIVE_CACHE else ""
    t0 = time.time()
    status[name] = {"state": "running", "started": time.strftime("%H:%M:%S")}
    save_status(status)

    if os.path.exists(marker):
        log(f"{name}: already complete at {dest}")
    elif CACHE_TO_DRIVE and cache_dir and os.path.exists(os.path.join(cache_dir, ".download_complete")):
        log(f"{name}: restoring from Drive cache {cache_dir}")
        shutil.copytree(cache_dir, dest, dirs_exist_ok=True)
    else:
        last_err = None
        used_repo = None
        for repo in [job["repo"]] + ([job["fallback_repo"]] if job.get("fallback_repo") else []):
            denied = False
            for attempt in range(1, 4):
                try:
                    log(f"{name}: downloading {repo} (attempt {attempt})")
                    snapshot_download(
                        repo_id=repo,
                        repo_type=job.get("repo_type", "model"),
                        local_dir=dest,
                        allow_patterns=job.get("allow"),
                        ignore_patterns=job.get("ignore"),
                        token=TOKEN,
                        max_workers=8,
                    )
                    last_err, used_repo = None, repo
                    break
                except Exception as e:  # network hiccups, gated repo, etc.
                    last_err = e
                    msg = str(e)
                    log(f"{name}: attempt {attempt} failed: {type(e).__name__}: {msg[:300]}")
                    if "401" in msg or "403" in msg or "gated" in msg.lower() or "access" in msg.lower():
                        denied = True
                        break  # retrying will not help with a permission problem
                    time.sleep(20 * attempt)
            if used_repo or not denied:
                break  # success, or a non-permission failure the fallback would not fix
            if job.get("fallback_repo") and repo != job["fallback_repo"]:
                log(f"{name}: no access to {repo}; trying fallback {job['fallback_repo']}")
        if used_repo:
            open(os.path.join(dest, ".source_repo"), "w").write(used_repo)
        if last_err is not None:
            status[name] = {"state": "failed", "error": f"{type(last_err).__name__}: {str(last_err)[:500]}",
                            "seconds": round(time.time() - t0)}
            save_status(status)
            return False
        open(marker, "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
        if CACHE_TO_DRIVE and cache_dir:
            log(f"{name}: copying to Drive cache {cache_dir}")
            shutil.copytree(dest, cache_dir, dirs_exist_ok=True)

    src_file = os.path.join(dest, ".source_repo")
    status[name] = {"state": "done", "path": dest, "size_gb": dir_size_gb(dest),
                    "repo": open(src_file).read() if os.path.exists(src_file) else job["repo"],
                    "seconds": round(time.time() - t0)}
    save_status(status)
    log(f"{name}: done ({status[name]['size_gb']} GB, {status[name]['seconds']} s)")
    return True


def main():
    os.makedirs(CKPT_DIR, exist_ok=True)
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(LOG_DIR, exist_ok=True)
    wanted = set(sys.argv[1:])
    jobs = [j for j in JOBS + OPTIONAL if j["name"] in wanted] if wanted else JOBS
    status = {}  # fresh per session: an old status file on Drive would show stale "done" entries
    for job in jobs:
        status.setdefault(job["name"], {"state": "pending"})
    save_status(status)
    ok = True
    for job in jobs:
        try:
            ok = run_job(job, status) and ok
        except Exception:
            traceback.print_exc()
            status[job["name"]] = {"state": "failed", "error": traceback.format_exc()[-500:]}
            save_status(status)
            ok = False
    log("ALL DONE" if ok else "FINISHED WITH FAILURES - see download_status.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/ellsa_scripts/inspect_assets.py
"""Inspect downloaded assets and prepare them for evaluation.

1. Prints the size of each downloaded folder.
2. Finds the actual ELLSA checkpoint folder (the one with config.json + weights) and
   reports which sub-models its weights cover, so we know whether the UniVLA weights
   must be downloaded as well.
3. Summarises the ELLSA test data layout.
4. Rewrites audio paths inside the test-data JSON files (they were written on the
   authors' servers) so they point at the local copy. Originals are kept as *.orig.json.
5. Writes the detected paths to $DETECTED_ENV (a bash file the notebook sources).

Usage: python inspect_assets.py
"""
import collections
import glob
import json
import os
import shutil

CKPT_DIR = os.environ["CKPT_DIR"]
DATA_DIR = os.environ["DATA_DIR"]
DETECTED_ENV = os.environ.get("DETECTED_ENV", "/content/ellsa_detected.env")
TEST_DATA = os.path.join(DATA_DIR, "ELLSA_test_data")
AUDIO_EXT = (".wav", ".flac", ".mp3", ".ogg")


def size_gb(path):
    total = 0
    for root, _, files in os.walk(path):
        for fn in files:
            fp = os.path.join(root, fn)
            if not os.path.islink(fp):
                total += os.path.getsize(fp)
    return total / 1e9


def section(title):
    print("\n" + "=" * 70 + f"\n{title}\n" + "=" * 70)


def find_ckpt_dir(root):
    """Folder that contains config.json and model weights (prefers the shallowest)."""
    cands = []
    for cfg in glob.glob(os.path.join(root, "**", "config.json"), recursive=True):
        d = os.path.dirname(cfg)
        if glob.glob(os.path.join(d, "*.safetensors")) or glob.glob(os.path.join(d, "*.bin")):
            cands.append(d)
    cands.sort(key=lambda p: p.count(os.sep))
    return cands


def weight_keys(ckpt):
    idx = glob.glob(os.path.join(ckpt, "*.index.json"))
    if idx:
        with open(idx[0]) as f:
            return list(json.load(f)["weight_map"].keys())
    keys = []
    try:
        from safetensors import safe_open
        for st in glob.glob(os.path.join(ckpt, "*.safetensors")):
            with safe_open(st, framework="pt") as f:
                keys += list(f.keys())
    except Exception as e:
        print("  could not open safetensors:", e)
    return keys


def fix_paths(obj, index, stats):
    """Recursively rewrite audio-path strings that do not exist locally."""
    if isinstance(obj, dict):
        return {k: fix_paths(v, index, stats) for k, v in obj.items()}
    if isinstance(obj, list):
        return [fix_paths(v, index, stats) for v in obj]
    if isinstance(obj, str) and obj.lower().endswith(AUDIO_EXT):
        stats["audio_refs"] += 1
        if os.path.exists(obj):
            stats["ok"] += 1
            return obj
        # 1) anything after ".../ELLSA_test_data/" maps onto our local copy
        marker = "ELLSA_test_data/"
        if marker in obj:
            cand = os.path.join(TEST_DATA, obj.split(marker, 1)[1])
            if os.path.exists(cand):
                stats["fixed"] += 1
                return cand
        # 2) longest matching path suffix among local files
        parts = obj.replace("\\", "/").split("/")
        for n in range(min(4, len(parts)), 0, -1):
            hits = index.get("/".join(parts[-n:]), [])
            if len(hits) == 1:
                stats["fixed"] += 1
                return hits[0]
        stats["unresolved"] += 1
        if len(stats["unresolved_examples"]) < 5:
            stats["unresolved_examples"].append(obj)
    return obj


def main():
    detected = {}

    section("1. Downloaded folder sizes")
    for base in (CKPT_DIR, DATA_DIR):
        for d in sorted(os.listdir(base)) if os.path.isdir(base) else []:
            p = os.path.join(base, d)
            if os.path.isdir(p):
                done = os.path.exists(os.path.join(p, ".download_complete"))
                print(f"  {size_gb(p):7.2f} GB  {'done' if done else 'INCOMPLETE'}  {p}")

    section("2. ELLSA checkpoint")
    cands = find_ckpt_dir(os.path.join(CKPT_DIR, "ELLSA"))
    if not cands:
        print("  !! no folder with config.json + weights found under", os.path.join(CKPT_DIR, "ELLSA"))
    else:
        ckpt = cands[0]
        detected["ELLSA_CKPT"] = ckpt
        print("  checkpoint folder:", ckpt, "(other candidates: %s)" % cands[1:] if len(cands) > 1 else "")
        print("  files:", sorted(os.listdir(ckpt))[:30])
        keys = weight_keys(ckpt)
        prefix = collections.Counter(".".join(k.split(".")[:2]) for k in keys)
        lora = sum("lora" in k for k in keys)
        print(f"  {len(keys)} tensors, {lora} LoRA tensors. Top prefixes:")
        for p, c in prefix.most_common(12):
            print(f"    {c:6d}  {p}")
        vis_base = sum(k.startswith("vision_expert.") and "lora" not in k for k in keys)
        sp_base = sum(k.startswith("speech_expert.") and "lora" not in k for k in keys)
        print(f"  vision_expert base tensors: {vis_base}   speech_expert base tensors: {sp_base}")
        if vis_base > 100:
            print("  VERDICT: vision expert weights are inside the ELLSA checkpoint -> UniVLA weights NOT needed.")
        else:
            print("  VERDICT: vision expert base weights missing -> run:  python download_weights.py UniVLA-weights")

    univla = os.path.join(CKPT_DIR, "UniVLA", "UNIVLA_LIBERO_VIDEO_BS192_8K")
    detected["UNIVLA_CKPT_PATH"] = univla
    print("\n  UniVLA config folder:", univla, "exists" if os.path.isdir(univla) else "MISSING")
    if os.path.isdir(univla):
        print("   ", sorted(os.listdir(univla))[:20])

    section("3. ELLSA test data")
    if not os.path.isdir(TEST_DATA):
        print("  !! test data missing:", TEST_DATA)
    else:
        for d in sorted(os.listdir(TEST_DATA)):
            p = os.path.join(TEST_DATA, d)
            if os.path.isdir(p):
                n = sum(len(f) for _, _, f in os.walk(p))
                print(f"  {d:30s} {n:6d} files")
            else:
                print(f"  {d}")
        detected["ELLSA_DATA_PATH"] = TEST_DATA

        section("4. Fixing audio paths in test-data JSON files")
        index = collections.defaultdict(list)
        for root, _, files in os.walk(TEST_DATA):
            for fn in files:
                if fn.lower().endswith(AUDIO_EXT):
                    full = os.path.join(root, fn)
                    parts = os.path.relpath(full, TEST_DATA).split(os.sep)
                    for n in range(1, min(4, len(parts)) + 1):
                        index["/".join(parts[-n:])].append(full)
        print(f"  {sum(1 for k in index if '/' not in k)} distinct local audio file names indexed")
        for jf in sorted(glob.glob(os.path.join(TEST_DATA, "**", "*.json"), recursive=True)):
            if jf.endswith(".orig.json"):
                continue
            orig = jf[:-5] + ".orig.json"
            if not os.path.exists(orig):
                shutil.copy(jf, orig)
            with open(orig) as f:
                data = json.load(f)
            stats = dict(audio_refs=0, ok=0, fixed=0, unresolved=0, unresolved_examples=[])
            new = fix_paths(data, index, stats)
            with open(jf, "w") as f:
                json.dump(new, f, ensure_ascii=False, indent=1)
            rel = os.path.relpath(jf, TEST_DATA)
            print(f"  {rel:40s} refs={stats['audio_refs']:5d} ok={stats['ok']:5d} "
                  f"fixed={stats['fixed']:5d} unresolved={stats['unresolved']}")
            for ex in stats["unresolved_examples"]:
                print("      unresolved e.g.", ex)
        key = os.path.join(TEST_DATA, "json", "libero_eval_speech.json")
        if os.path.exists(key):
            d = json.load(open(key))
            print("\n  libero_eval_speech.json suites:", list(d.keys()))
            if "libero_goal" in d:
                g = d["libero_goal"]
                print("  libero_goal task 0 ->", g.get("0"), "exists:", os.path.exists(str(g.get("0"))))

    with open(DETECTED_ENV, "w") as f:
        for k, v in detected.items():
            f.write(f'export {k}="{v}"\n')
    section(f"Detected paths written to {DETECTED_ENV}")
    print(open(DETECTED_ENV).read())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/ellsa_scripts/patch_eval.py
"""Create evaluate_libero_smoke.py: a copy of ELLSA's official LIBERO evaluation script
with small, additive changes. The original file is never modified.

Changes
  * --num_trials N        episodes per task (official script hard-codes 50)
  * --task_ids 0,3        evaluate only these task ids (default: all)
  * --results_jsonl PATH  one JSON line per episode: success, steps, seconds, peak GPU memory
  * `import kaldifeat` made optional (only used by encoder_type=mamba; ELLSA uses zipformer2)

Usage: python patch_eval.py <ELLSA repo root>
"""
import os
import sys

root = sys.argv[1] if len(sys.argv) > 1 else os.environ["ELLSA_BASE_PATH"]
src = os.path.join(root, "reference/RoboVLMs/eval/libero/evaluate_libero_emu_contemporary.py")
dst = os.path.join(root, "reference/RoboVLMs/eval/libero/evaluate_libero_smoke.py")
code = open(src).read()

PATCHES = [
    # 1. optional kaldifeat
    ("import kaldifeat\n",
     "try:\n    import kaldifeat\nexcept ImportError:  # only needed for encoder_type=mamba\n    kaldifeat = None\n"),
    # 2. new evaluate() arguments
    ("    predict_action_frames=10,\n    generate=False\n):",
     "    predict_action_frames=10,\n    generate=False,\n    task_ids=None,\n    results_jsonl=None\n):"),
    # 3. task filter
    ("    for task_id in range(num_tasks_in_suite):\n",
     "    for task_id in range(num_tasks_in_suite):\n"
     "        if task_ids is not None and task_id not in task_ids:\n"
     "            continue\n"),
    # 4. per-episode timer / memory counter
    ("            env.reset()\n            model.reset()\n",
     "            env.reset()\n            model.reset()\n"
     "            ep_t0 = time.time()\n"
     "            if torch.cuda.is_available():\n"
     "                torch.cuda.reset_peak_memory_stats()\n"),
    # 5. per-episode JSON line
    ('            log_file.write(f"Success: {current_done}\\n")\n',
     '            log_file.write(f"Success: {current_done}\\n")\n'
     "            if results_jsonl:\n"
     "                with open(results_jsonl, 'a') as _rf:\n"
     "                    _rf.write(json.dumps({\n"
     "                        'suite': task_suite_name, 'task_id': task_id, 'task': task_description,\n"
     "                        'episode_idx': episode_idx, 'success': bool(current_done), 'steps': int(t),\n"
     "                        'seconds': round(time.time() - ep_t0, 2),\n"
     "                        'peak_gpu_gb': round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None,\n"
     "                        'hyp': hyp}) + '\\n')\n"),
    # 6. CLI flags
    ("    args = parser.parse_args()\n",
     "    parser.add_argument('--num_trials', type=int, default=50, help='episodes per task')\n"
     "    parser.add_argument('--task_ids', type=str, default='', help='comma separated task ids, empty = all')\n"
     "    parser.add_argument('--results_jsonl', type=str, default='', help='per-episode results file')\n"
     "    args = parser.parse_args()\n"),
    # 7. pass them through
    ("        generate=args.generate\n    )",
     "        generate=args.generate,\n"
     "        num_trials_per_task=args.num_trials,\n"
     "        task_ids=[int(x) for x in args.task_ids.split(',') if x.strip()] or None,\n"
     "        results_jsonl=args.results_jsonl or None\n    )"),
]

for old, new in PATCHES:
    n = code.count(old)
    if n != 1:
        sys.exit(f"PATCH FAILED: expected exactly 1 match, found {n} for:\n{old}")
    code = code.replace(old, new)

with open(dst, "w") as f:
    f.write(code)
print("wrote", dst)

In [ ]:
%%writefile /content/ellsa_scripts/libero_render_check.py
"""Smallest possible LIBERO check: build one LIBERO-Goal environment, step it, save a frame.

The rendering backend is chosen by MUJOCO_GL (egl on a GPU runtime, osmesa as a CPU fallback);
it must be set before this script starts, so the notebook calls it once per backend.

Usage: MUJOCO_GL=egl python libero_render_check.py <output_png>
Exit code 0 = success.
"""
import os
import sys
import time

import numpy as np
from PIL import Image
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

out_png = sys.argv[1] if len(sys.argv) > 1 else "libero_check.png"
t0 = time.time()
suite = benchmark.get_benchmark_dict()["libero_goal"]()
task = suite.get_task(0)
env = OffScreenRenderEnv(
    bddl_file_name=os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file),
    camera_heights=256, camera_widths=256)
env.seed(0)
env.reset()
obs = env.set_init_state(suite.get_task_init_states(0)[0])
build_s = time.time() - t0
t1 = time.time()
for _ in range(20):
    obs, reward, done, info = env.step([0, 0, 0, 0, 0, 0, -1])
step_ms = (time.time() - t1) / 20 * 1000
img = np.concatenate([obs["agentview_image"][::-1, ::-1], obs["robot0_eye_in_hand_image"][::-1, ::-1]], axis=1)
Image.fromarray(img).save(out_png)
print(f"OK backend={os.environ.get('MUJOCO_GL')} task='{task.language}' "
      f"build={build_s:.1f}s step={step_ms:.1f}ms image={out_png}")
env.close()

In [ ]:
%%writefile /content/ellsa_scripts/colab_requirements.txt
transformers==4.44.0
tiktoken==0.6.0
peft==0.16.0
pillow
gradio==4.44.0
numpydantic==1.6.8
opencv-python==4.11.0.86
lhotse==1.30.3
moviepy==2.2.1
lightning==2.6.1
pytorch-lightning==2.6.1
HyperPyYAML==1.2.3
onnxruntime==1.23.2
openai-whisper==20250625
inflect
wetextprocessing
omegaconf==2.3.0
scipy==1.15.3
conformer==0.3.2
diffusers==0.29.0
hydra-core==1.3.2
pyarrow==18.1.0
librosa==0.11.0
einops
regex
soundfile
imageio[ffmpeg]
termcolor
robosuite==1.4.0
bddl
easydict
cloudpickle
gym
h5py
matplotlib
numpy==1.26.4
mujoco==2.3.7

In [ ]:
#@title A5. ▶ 가중치 다운로드 시작 (백그라운드)
import subprocess, os
log = open(os.environ["LOG_DIR"] + "/download.log", "a")
proc = subprocess.Popen(["python3", os.environ["SCRIPTS"] + "/download_weights.py"],
                        stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=os.environ.copy())
print(f"다운로드 시작 (pid {proc.pid}). 진행 상황은 A6 셀을 다시 실행해서 확인하세요.")
print("순서: ELLSA_test_data → ELLSA → Emu3-VisionTokenizer → UniVLA(config) → CosyVoice2 → Llama-3.1-8B")
print("이제 파트 B로 넘어가세요. 설치는 다운로드와 동시에 진행됩니다.")

In [ ]:
#@title A6. 다운로드 진행 상황 (여러 번 실행해도 됨)
import json, os, subprocess
st_path = os.environ["LOG_DIR"] + "/download_status.json"
if os.path.exists(st_path):
    for name, s in json.load(open(st_path)).items():
        icon = {"done": "✅", "running": "⏳", "failed": "❌", "pending": "·"}.get(s.get("state"), "?")
        extra = f"{s.get('size_gb','')} GB {s.get('seconds','')} s  [{s.get('repo','')}]" if s.get("state") == "done" else s.get("error", "")[:160]
        print(f"{icon} {name:24s} {s.get('state'):8s} {extra}")
print("\n--- local sizes ---")
print(subprocess.run("du -sh /content/ckpt/* /content/data/* 2>/dev/null", shell=True, capture_output=True, text=True).stdout)
print("--- last log lines ---")
print(subprocess.run(f"tail -n 6 {os.environ['LOG_DIR']}/download.log", shell=True, capture_output=True, text=True).stdout)

## 파트 B. Python 3.10 환경 + 코드 설치 (다운로드와 동시에 진행)
Colab 기본 Python은 3.10이 아니므로 `/content/venv`에 별도 환경을 만듭니다.
이후 모든 bash 셀은 `source /content/ellsa_env.sh`로 시작합니다.

In [ ]:
%%bash
#@title B1. 시스템 패키지 + Python 3.10 + 의존성 설치 (10–20분)
# Safe to re-run: finished steps are quick no-ops. On failure it prints the failing step and the log tail.
SYS_PY=$(command -v python3)          # Colab's own Python (before the venv is put on PATH)
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 설정 파일 또는 Drive 연결이 없습니다. 파트 A의 A2 셀(그리고 A3·A4·A5)을 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
mkdir -p "$LOG_DIR" "$RESULTS_DIR"
LOG=$LOG_DIR/install.log; echo "===== $(date) =====" >> $LOG
step() {
  local name="$1"; shift
  printf "%-28s" "$name"
  if "$@" >>$LOG 2>&1; then echo "OK"
  else echo "FAILED"; echo "----- last 30 lines of $LOG -----"; tail -n 30 $LOG
       echo "-----------------------------------------------"; echo "↑ 이 내용을 그대로 복사해서 보내주세요."; exit 1; fi
}
apt-get -qq update >>$LOG 2>&1 || echo "(apt update reported warnings - continuing)"
step "[1/6] apt EGL/OSMesa"     apt-get -qq install -y libegl1 libgl1 libosmesa6 libglib2.0-0
step "[2/6] uv"                 $SYS_PY -m pip install -q uv
step "      python 3.10"        env -u UV_SYSTEM_PYTHON uv python install 3.10
if [ -x $VENV/bin/python ] && $VENV/bin/python -c "import sys; assert sys.version_info[:2]==(3,10)" 2>/dev/null; then
  echo "      venv                exists, reusing"
else
  rm -rf $VENV; step "      venv" env -u UV_SYSTEM_PYTHON uv venv --python 3.10 $VENV
fi
# Colab sets UV_SYSTEM_PYTHON=1, which makes uv ignore the venv and target Colab's Python 3.13.
unset UV_SYSTEM_PYTHON UV_PYTHON
export VIRTUAL_ENV=$VENV
VPY="--python $VENV/bin/python"
step "[3/6] torch 2.4.0 cu124"  uv pip install $VPY torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124
step "[4/6] ELLSA/LIBERO deps"  uv pip install $VPY -r $SCRIPTS/colab_requirements.txt setuptools wheel huggingface_hub
step "[5/6] deepspeed"          uv pip install $VPY deepspeed==0.18.8 --no-build-isolation
step "[6/6] flash-attn wheel"   uv pip install $VPY "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.4cxx11abiFALSE-cp310-cp310-linux_x86_64.whl"
$VENV/bin/python -c "import sys, torch; print('venv python', sys.version.split()[0], '| torch', torch.__version__)"
echo "✅ DONE (log: $LOG)"

In [ ]:
%%bash
#@title B2. 설치 확인
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 설정 파일 또는 Drive 연결이 없습니다. 파트 A의 A2 셀(그리고 A3·A4·A5)을 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
mkdir -p "$LOG_DIR" "$RESULTS_DIR"
python - <<'EOF' 2>&1 | grep -v -i "warning\|gym\|migration"
import sys, importlib, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
bad = []
for m in ["flash_attn", "transformers", "peft", "deepspeed", "lightning", "pytorch_lightning", "lhotse", "whisper",
          "hyperpyyaml", "onnxruntime", "tn.english.normalizer", "diffusers", "conformer", "einops",
          "robosuite", "bddl", "mujoco", "cv2", "numpy"]:
    try:
        mod = importlib.import_module(m); print(f"  OK  {m} {getattr(mod, '__version__', '')}")
    except Exception as e:
        bad.append(m); print(f"  ERR {m}: {type(e).__name__}: {str(e)[:150]}")
print("\n✅ all imports OK" if not bad else f"\n❌ failed: {bad}  (see $LOG_DIR/install.log)")
EOF

In [ ]:
%%bash
#@title B3. ELLSA·LIBERO 코드 (커밋 고정) + LIBERO 설정 + 평가 스크립트 패치
set -e
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 설정 파일 또는 Drive 연결이 없습니다. 파트 A의 A2 셀(그리고 A3·A4·A5)을 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
mkdir -p "$LOG_DIR" "$RESULTS_DIR"
if [ ! -d $ELLSA_BASE_PATH ]; then
  git clone -q -b ELLSA https://github.com/bytedance/SALMONN.git $ELLSA_BASE_PATH
  git -C $ELLSA_BASE_PATH fetch -q --depth 1 origin b2d6d1a07c8b402287da8c4fe58edec7a4c10d08 2>/dev/null || true
  git -C $ELLSA_BASE_PATH checkout -q b2d6d1a07c8b402287da8c4fe58edec7a4c10d08 2>/dev/null || echo "(note: pinned ELLSA commit not found, using branch head)"
fi
if [ ! -d $LIBERO_ROOT ]; then
  git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git $LIBERO_ROOT
  git -C $LIBERO_ROOT checkout -q 8f1084e3132a39270c3a13ebe37270a43ece2a01 2>/dev/null || echo "(note: pinned LIBERO commit not found, using head)"
fi
echo "ELLSA  commit: $(git -C $ELLSA_BASE_PATH rev-parse --short HEAD)"
echo "LIBERO commit: $(git -C $LIBERO_ROOT rev-parse --short HEAD)"

# LIBERO asks an interactive question on first import unless this file exists
mkdir -p ~/.libero
cat > ~/.libero/config.yaml <<EOF
benchmark_root: $LIBERO_ROOT/libero/libero
bddl_files: $LIBERO_ROOT/libero/libero/bddl_files
init_states: $LIBERO_ROOT/libero/libero/init_files
datasets: $LIBERO_ROOT/libero/datasets
assets: $LIBERO_ROOT/libero/libero/assets
EOF

python $SCRIPTS/patch_eval.py $ELLSA_BASE_PATH

In [ ]:
%%bash
#@title B4. LIBERO 렌더링 점검 (EGL → 실패 시 OSMesa)
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 설정 파일 또는 Drive 연결이 없습니다. 파트 A의 A2 셀(그리고 A3·A4·A5)을 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
mkdir -p "$LOG_DIR" "$RESULTS_DIR"
for BK in egl osmesa; do
  if MUJOCO_GL=$BK PYOPENGL_PLATFORM=$BK timeout 600 python $SCRIPTS/libero_render_check.py /content/libero_check.png 2>&1 | grep -v -i "warning\|gym\|migration\|macro" | tail -3 | tee /tmp/render.out | grep -q "^OK"; then
    cat /tmp/render.out
    printf 'export MUJOCO_GL=%s\nexport PYOPENGL_PLATFORM=%s\n' $BK $BK > /content/ellsa_render.env
    echo "✅ rendering backend: $BK"; break
  else
    cat /tmp/render.out; echo "❌ $BK failed"
  fi
done

In [ ]:
#@title B5. 렌더링 결과 보기 (왼쪽 정면 카메라, 오른쪽 손목 카메라)
from IPython.display import Image, display
display(Image("/content/libero_check.png"))

In [ ]:
%%bash
#@title B6. ELLSA 평가 코드 import 점검 (모델은 아직 로드하지 않음)
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 설정 파일 또는 Drive 연결이 없습니다. 파트 A의 A2 셀(그리고 A3·A4·A5)을 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
mkdir -p "$LOG_DIR" "$RESULTS_DIR"
cd $ELLSA_BASE_PATH/reference/RoboVLMs
python eval/libero/evaluate_libero_smoke.py --help > /tmp/help.txt 2>&1 \
  && echo "✅ import chain OK (LIBERO, CosyVoice, Emu3, model wrapper)" \
  || (grep -v -i "warning\|gym\|migration" /tmp/help.txt | tail -20; echo "❌ import failed")

## 파트 C. 다운로드 완료 후: 점검 → 스모크 테스트
A6에서 모든 항목이 ✅인지 먼저 확인하세요. Llama 줄의 `[...]`에 실제로 받은 저장소(공식 또는 unsloth 미러)가 표시됩니다.

In [ ]:
%%bash
#@title C1. 다운로드 결과 점검 + 테스트 데이터 경로 보정
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 설정 파일 또는 Drive 연결이 없습니다. 파트 A의 A2 셀(그리고 A3·A4·A5)을 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
mkdir -p "$LOG_DIR" "$RESULTS_DIR"
python $SCRIPTS/inspect_assets.py 2>&1 | tee $LOG_DIR/inspect_assets.log

In [ ]:
%%bash
#@title C2. (필요할 때만) 개별 다운로드 재시도
# C1이 "UniVLA weights NEEDED"라고 하면: JOB="UniVLA-weights"
# Llama 접근 승인 후 재시도:           JOB="Llama-3.1-8B-Instruct"
JOB="Llama-3.1-8B-Instruct"
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 설정 파일 또는 Drive 연결이 없습니다. 파트 A의 A2 셀(그리고 A3·A4·A5)을 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
mkdir -p "$LOG_DIR" "$RESULTS_DIR"
python3 $SCRIPTS/download_weights.py $JOB 2>&1 | tail -5

### C3. 스모크 테스트
- 설정: **LIBERO-Goal 과제 0 ("open the middle drawer of the cabinet") × 2 에피소드**, 음성 지시 조건(논문 Table 2와 같은 `--silence True` 설정)
- 결과는 Drive `ellsa/results/smoke_*`에 저장됩니다: `episodes.jsonl`(에피소드별 결과), `stdout.log`, 에피소드 GIF
- ⚠️ 이 스크립트의 bool 옵션은 `type=bool`이라 `--generate False`도 **True로 해석됩니다**. 끄려면 옵션 자체를 빼야 합니다. 스모크 테스트에서는 음성 출력이 필요 없어 `--generate`를 넣지 않았습니다. 메모리와 시간이 절약됩니다.
- 첫 실행은 모델 로딩에 수 분이 걸립니다.

In [ ]:
%%bash
#@title C3. ▶ 스모크 테스트 실행
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 설정 파일 또는 Drive 연결이 없습니다. 파트 A의 A2 셀(그리고 A3·A4·A5)을 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
mkdir -p "$LOG_DIR" "$RESULTS_DIR"
cd $ELLSA_BASE_PATH/reference/RoboVLMs
RUN=$RESULTS_DIR/smoke_$(date +%m%d_%H%M); mkdir -p $RUN; echo $RUN > /content/last_run.txt
echo "ckpt: $ELLSA_CKPT"; echo "out : $RUN"
nvidia-smi --query-gpu=memory.used --format=csv,noheader -lms 5000 > $RUN/gpu_mem.csv &
MON=$!
python eval/libero/evaluate_libero_smoke.py \
  --emu_hub "$ELLSA_CKPT" --vision_hub "$VISION_VQ_PATH" \
  --no_nccl --no_action_ensemble \
  --task_suite_name libero_goal --task_ids 0 --num_trials 2 \
  --speech True --moe True --merge_speech_lora True --lora_modules qkv \
  --encoder_type zipformer2 --silence True \
  --cache_root "$RUN" --results_jsonl "$RUN/episodes.jsonl" --debug \
  > $RUN/stdout.log 2>&1
EXIT=$?
kill $MON 2>/dev/null
grep -E "Task:|Starting episode|Success:|# successes|action end|Caught exception|Error|error:|Traceback|OutOfMemory" $RUN/stdout.log | tail -30
echo "exit code: $EXIT   (full log: $RUN/stdout.log)"

In [ ]:
#@title C4. 결과 요약 + 전체 평가 소요 시간 추정
import json, os, glob
from IPython.display import Image, display
run = open("/content/last_run.txt").read().strip()
eps = [json.loads(l) for l in open(f"{run}/episodes.jsonl")] if os.path.exists(f"{run}/episodes.jsonl") else []
if not eps:
    print("에피소드 결과가 없습니다. stdout.log를 확인하세요:", f"{run}/stdout.log")
else:
    for e in eps:
        print(f"task {e['task_id']} ep {e['episode_idx']}: success={e['success']} steps={e['steps']} "
              f"{e['seconds']} s  peak GPU {e['peak_gpu_gb']} GB")
    sr = sum(e["success"] for e in eps) / len(eps)
    sec = sum(e["seconds"] for e in eps) / len(eps)
    mem = [int(x.split()[0]) for x in open(f"{run}/gpu_mem.csv") if x.strip()] if os.path.exists(f"{run}/gpu_mem.csv") else []
    print(f"\nsuccess rate {sr:.0%} | {sec:.0f} s/episode | nvidia-smi peak {max(mem)/1024 if mem else float('nan'):.1f} GB")
    units_per_hour = 7.52  #@param {type:"number"}
    # A100 80GB ≈ 7.52 compute units/h (2026-03 기준). 런타임 화면의 값으로 바꿔도 됩니다.
    for label, n in [("논문 재현 전체 (4 suite × 500)", 2000), ("축소 재현 (4 suite × 100)", 400),
                     ("LIBERO-Goal 1 suite × 100", 100)]:
        h = n * sec / 3600
        print(f"  {label:32s} {n:5d} ep → {h:5.1f} h ≈ {h*units_per_hour:5.0f} compute units")
gifs = sorted(glob.glob(f"{run}/**/*.gif", recursive=True))
print("\nGIFs:", gifs)
if gifs:
    display(Image(gifs[0]))

## 막혔을 때

| 증상 | 원인 / 해결 |
|---|---|
| B1 실패 | `ellsa/logs/install.log` 마지막 부분 확인. 대부분 네트워크 일시 오류이므로 B1 재실행 |
| B4에서 egl·osmesa 모두 실패 | B1의 apt 단계 확인. `apt-get install -y libegl1 libgl1 libosmesa6` 수동 실행 |
| C3 `OutOfMemoryError` | 40GB GPU가 배정됨 → 런타임을 A100 80GB/H100으로 변경 |
| A6에서 Llama ❌ | 공식·미러 모두 실패한 경우. 네트워크 문제일 가능성이 높으니 C2로 재시도 |
| C3 `FileNotFoundError ... .wav` | C1의 `unresolved` 항목 확인 (테스트 데이터 경로 보정 실패) |
| C3 가중치 관련 경고가 대량으로 나오고 성공률 0% | C1 VERDICT 확인 → `UniVLA-weights` 다운로드 (C2) |
| 런타임이 끊김 | 파트 A, B를 다시 실행. 이미 받은 파일은 다시 받지 않음 (같은 세션 안에서). `CACHE_TO_DRIVE=True`였다면 Drive에서 복원 |

**C4 결과(성공 여부, 에피소드당 시간, 최대 GPU 메모리)를 기록해 두세요.** 다음 단계인 Switch 평가셋 설계와 컴퓨트 예산 계산에 바로 쓰입니다.

## 파트 D. SwitchBench: 평가셋 생성 + ELLSA 베이스라인 (B0)
- **D1** 챌린지 코드 설치 → **D2** 평가 음성 생성(dev/test, 5–10분) → **D3** 베이스라인 평가를 백그라운드로 시작 → **D4** 진행 상황·중간 점수 (여러 번 실행)
- dev 50 에피소드(전환 30, 유지 10,일반 10). 예상 2.5–3.5시간. 결과는 에피소드마다 Drive에 저장되고, 끊기면 D3을 다시 실행해서 이어서 돌립니다.

In [ ]:
#@title D1. SwitchBench 코드 설치
import base64, io, tarfile, os, shutil
B64 = "..."
shutil.rmtree("/content/switchbench", ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64))).extractall("/content")
for r, d, f in os.walk("/content/switchbench"):
    d[:] = [x for x in d if x != "__pycache__"]
    print(r.replace("/content/", ""), sorted(f))

In [ ]:
%%bash
#@title D2. 평가셋 음성 구성 (ELLSA 테스트 데이터의 공식 녹음 사용)
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 파트 A(A2)를 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
export PYTHONPATH="$PYTHONPATH:/content/switchbench"
SB_DATA=$DRIVE_ROOT/switchbench_data
B0_OUT=$RESULTS_DIR/b0_ellsa_dev

# Official recordings are in ELLSA's training distribution. Google TTS voices made ELLSA reject valid
# instructions ("Action cancelled"), which would confound the switching measurement.
cd /content/switchbench
for SPLIT in dev test; do
  echo "===== $SPLIT ====="
  python scripts/build_dataset.py --split $SPLIT --out $SB_DATA --engine official --prompt_dir $ELLSA_DATA_PATH \
    2>&1 | grep -v -i "warn\|gym\|migration" | tail -n 7
done

In [ ]:
#@title D2-1. (선택) 생성된 음성 들어보기
import json, os, IPython.display as ipd
root = os.environ["DRIVE_ROOT"] + "/switchbench_data/dev"
eps = json.load(open(root + "/manifest.json"))["episodes"]
for e in eps[:4]:
    print(e["id"], "|", e["instruction_text"], "|", e["utterance_text"])
    ipd.display(ipd.Audio(root + "/" + e["instruction_audio"]))
    if e["utterance_audio"]: ipd.display(ipd.Audio(root + "/" + e["utterance_audio"]))

In [ ]:
%%bash
#@title D3. ▶ ELLSA 베이스라인(B0) 평가 시작 (백그라운드, 재실행하면 이어서 진행)
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 파트 A(A2)를 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
export PYTHONPATH="$PYTHONPATH:/content/switchbench"
SB_DATA=$DRIVE_ROOT/switchbench_data
B0_OUT=$RESULTS_DIR/b0_ellsa_dev

cd /content/switchbench
mkdir -p $B0_OUT
if pgrep -f "run_eval.py --data $SB_DATA/dev --policy ellsa" > /dev/null; then echo "이미 실행 중입니다. D4로 확인하세요."; exit 0; fi
setsid nohup python scripts/run_eval.py --data $SB_DATA/dev --policy ellsa --out $B0_OUT --videos 10 \
  >> $B0_OUT/run.log 2>&1 < /dev/null &
sleep 5; echo "started (pid $!). 모델 로딩에 3–5분 걸립니다. D4로 진행 상황을 확인하세요."
echo "log: $B0_OUT/run.log"

In [ ]:
%%bash
#@title D4. 진행 상황 + 중간 점수 (여러 번 실행)
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 파트 A(A2)를 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
export PYTHONPATH="$PYTHONPATH:/content/switchbench"
SB_DATA=$DRIVE_ROOT/switchbench_data
B0_OUT=$RESULTS_DIR/b0_ellsa_dev

cd /content/switchbench
pgrep -f "run_eval.py --data $SB_DATA/dev --policy ellsa" > /dev/null && echo "⏳ running" || echo "■ not running"
nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader
echo "--- recent log ---"
grep -E "^\[|ready in|already done|Error|Traceback|OutOfMemory" $B0_OUT/run.log | tail -n 8
if [ -s $B0_OUT/episodes.jsonl ]; then echo; python scripts/score.py $B0_OUT/episodes.jsonl; fi

In [ ]:
#@title D5. 에피소드 영상 보기
import glob, os
from IPython.display import Image, display
gifs = sorted(glob.glob(os.environ["RESULTS_DIR"] + "/b0_ellsa_dev/videos/*.gif"))
print(len(gifs), "videos")
for g in gifs[:4]:
    print(os.path.basename(g)); display(Image(g))

## 파트 E. 학습 없는 경량 방법: P1(의도 판별 재시작)과 B1(무조건 재시작)
- **P1**: 실행 중 새 발화를 Whisper로 받아적고, 조작 명령일 때만 ELLSA를 그 발화 기준으로 재시작. 질문이면 그대로 둠.
- **B1**: 실행 중 새 발화가 들리면 무조건 재시작 (비교 기준).
- E1이 P1(dev 50개) → B1(전환·유지 40개) 순서로 백그라운드 실행합니다. 합계 약 3.5–4.5시간. Score = (SSR+KSR)/2 라서 B1은 일반(plain) 에피소드를 생략합니다(두 번째 발화가 없어 B0와 동작이 같음).
- 사전 조건: 파트 A, B, C1, D1 완료. 평가 데이터는 Drive에 이미 있으므로 D2는 실행하지 않습니다.

In [ ]:
%%bash
#@title E1. ▶ P1 → B1 평가 시작 (백그라운드)
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 파트 A(A2)를 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
export PYTHONPATH="$PYTHONPATH:/content/switchbench"
SB_DATA=$DRIVE_ROOT/switchbench_data
B0_OUT=$RESULTS_DIR/b0_ellsa_dev

cd /content/switchbench
if pgrep -f "run_eval.py" > /dev/null; then echo "이미 평가가 실행 중입니다. E2로 확인하세요."; exit 0; fi
[ -f $SB_DATA/dev/manifest.json ] || { echo "❌ $SB_DATA/dev/manifest.json 없음 (Drive 연결 확인)"; exit 1; }
mkdir -p $RESULTS_DIR/p1_dev $RESULTS_DIR/b1_dev
echo '{"name": "P1: intent-gated restart (Whisper + rule)", "kind": "proposed", "training": "none"}' > $RESULTS_DIR/p1_dev/meta.json
echo '{"name": "B1: restart on any speech", "kind": "baseline", "training": "none", "note": "switch+keep only"}' > $RESULTS_DIR/b1_dev/meta.json
[ -f $B0_OUT/meta.json ] || echo '{"name": "B0: ELLSA (zero-shot)", "kind": "baseline", "training": "none"}' > $B0_OUT/meta.json
cat > /content/run_e.sh <<EOF
cd /content/switchbench
python scripts/run_eval.py --data $SB_DATA/dev --policy p1 --out $RESULTS_DIR/p1_dev --videos 6 >> $RESULTS_DIR/p1_dev/run.log 2>&1
python scripts/run_eval.py --data $SB_DATA/dev --policy b1 --out $RESULTS_DIR/b1_dev --videos 6 --types switch,keep >> $RESULTS_DIR/b1_dev/run.log 2>&1
EOF
setsid nohup bash /content/run_e.sh > /dev/null 2>&1 < /dev/null &
sleep 5; echo "✅ started: P1 then B1. 모델 로딩 3–5분 뒤부터 E2에서 결과가 보입니다."

In [ ]:
%%bash
#@title E2. 진행 상황 + 리더보드 (여러 번 실행)
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 파트 A(A2)를 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
export PYTHONPATH="$PYTHONPATH:/content/switchbench"
SB_DATA=$DRIVE_ROOT/switchbench_data
B0_OUT=$RESULTS_DIR/b0_ellsa_dev

cd /content/switchbench
pgrep -f "run_eval.py" > /dev/null && echo "⏳ running: $(pgrep -af run_eval.py | grep -o 'policy [a-z0-9]*' | head -1)" || echo "■ not running"
for P in p1 b1 p2; do
  echo "--- $P recent log ---"
  grep -E "^\[|ready in|Error|Traceback|OutOfMemory" $RESULTS_DIR/${P}_dev/run.log 2>/dev/null | tail -n 4
done
echo; echo "--- what P1 heard and decided (first 8) ---"
grep -ho '\[p1_intent_gated_reset\][^|]*' $RESULTS_DIR/p1_dev/episodes.jsonl 2>/dev/null | head -n 8
echo
RUNS="$B0_OUT"; for P in p1 b1 p2; do [ -s $RESULTS_DIR/${P}_dev/episodes.jsonl ] && RUNS="$RUNS $RESULTS_DIR/${P}_dev"; done
python scripts/leaderboard.py $RUNS --out $RESULTS_DIR/leaderboard | sed 's/^/  /'

In [ ]:
%%bash
#@title E3. ▶ P2 평가 (SwitchBench_P2 노트북에서 학습한 판별기 + ELLSA)
if [ ! -f /content/ellsa_env.sh ] || [ ! -d /content/drive/MyDrive ]; then echo "❌ 파트 A(A2)를 먼저 실행하세요."; exit 1; fi
source /content/ellsa_env.sh
export PYTHONPATH="$PYTHONPATH:/content/switchbench"
SB_DATA=$DRIVE_ROOT/switchbench_data
B0_OUT=$RESULTS_DIR/b0_ellsa_dev

export SWITCHBENCH_GATE=$RESULTS_DIR/gate_whisper-base/gate.pt
[ -f $SWITCHBENCH_GATE ] || { echo "❌ gate not found: run SwitchBench_P2.ipynb (F2) first"; exit 1; }
if pgrep -f "run_eval.py" > /dev/null; then echo "다른 평가가 실행 중입니다. 끝난 뒤 다시 실행하세요."; exit 0; fi
mkdir -p $RESULTS_DIR/p2_dev
echo '{"name": "P2: learned intent gate + restart", "kind": "proposed", "training": "SwitchBench-train (gate only)"}' > $RESULTS_DIR/p2_dev/meta.json
cd /content/switchbench
setsid nohup python scripts/run_eval.py --data $SB_DATA/dev --policy p2 --out $RESULTS_DIR/p2_dev --videos 6 \
  >> $RESULTS_DIR/p2_dev/run.log 2>&1 < /dev/null &
sleep 5; echo "✅ P2 evaluation started (~2 h). E2 shows progress and the leaderboard."